# Interactive Sequencing Exploration

This notebook is a lightweight interactive version of `hooke01_run_wt_regression_bead.R`.

It is intended for:
- loading one projected `cds` dataset at a time
- inspecting cell metadata
- plotting UMAP coordinates interactively
- building embryo-level cell count tables
- running simple regressions on metadata or counts


In [ ]:
library(monocle3)
library(dplyr)
library(tidyr)
library(plotly)
library(hooke)
library(BPCells)
library(ggplot2)

Loading required package: Biobase

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, setdiff, table,
    tapply, union, unique, unsplit, which.max, which.min


Welcome to Bioconductor

    Vignettes contain introductory material; view with
    'browseVignettes()'. To cite Bioconductor, see
    'citation("Biobase")', and for packages 'citation("pkgname")'.


Loading required package: SingleCellExperiment

Loading required package: SummarizedExperiment

Loading required package: MatrixGenerics

Loading requi

In [2]:
dyn.load("/net/gs/vol3/software/modules-sw/hdf5/1.14.3/Linux/Ubuntu22.04/x86_64/lib/libhdf5.so.310")
library(BPCells)

In [3]:
temp_path <- "/net/trapnell/vol1/home/nlammers/tmp_files/nobackup/"
seahub_root <- "/net/seahub_zfish/vol1/data/annotated/v2.2.0/"
ct_path <- "/net/trapnell/vol1/home/elizab9/projects/projects/CHEMFISH/resources/unique_ct_full.csv"

dataset_name <- "REF1"
use_hotfish2 <- FALSE

hot_path <- "/net/trapnell/vol1/home/hklee206/sci_3lvl/240813_hotfish2_run3_novaseq/hotfish2/hotfish2_projected_cds_v2.2.0"
cds_path <- file.path(seahub_root, dataset_name, paste0(dataset_name, "_projected_cds_v2.2.0/"))

## Load broad cell type mapping

In [4]:
ct_broad <- read.csv(ct_path)
ct_broad_filt <- as.data.frame(ct_broad) %>%
  count(cell_type, cell_type_broad) %>%
  group_by(cell_type) %>%
  slice_max(n, with_ties = FALSE) %>%
  ungroup() %>%
  select(cell_type, cell_type_broad)

head(ct_broad_filt)

cell_type,cell_type_broad
<chr>,<chr>
HR ionocyte,HR ionocyte
KA neuron,KA neuron
KS ionocyte,KS ionocyte
"MHB, progenitor","MHB, progenitor"
"MHB, progenitor (suspected doublets)","MHB, progenitor"
NaR ionocyte,NaR ionocyte


## Load one dataset interactively

In [5]:
if (use_hotfish2) {
  cds <- load_monocle_objects(
    hot_path,
    matrix_control = list(matrix_class = "BPCells", matrix_path = temp_path)
  )
} else {
  cds <- load_monocle_objects(
    cds_path,
    matrix_control = list(matrix_class = "BPCells", matrix_path = temp_path)
  )
}

cds

class: cell_data_set 
dim: 32031 739688 
metadata(1): cds_version
assays(2): counts counts_row_order
rownames(32031): ENSDARG00000000001 ENSDARG00000000002 ...
  ENSDARG00000117826 ENSDARG00000117827
rowData names(6): gene_short_name id ... bp2 gene_strand
colnames(739688): E01_A01_P05-A01_LIG14 E01_A01_P05-A01_LIG15 ...
  H02_G11_P05-G12_LIG59 H02_G11_P07-G11_LIG59
colData names(51): cell Size_Factor ... tissue cell_type_broad
reducedDimNames(9): PCA Aligned ... sub_Aligned sub_UMAP
mainExpName: NULL
altExpNames(0):

In [ ]:
md <- as.data.frame(colData(cds))

if (!('cell_type_broad' %in% colnames(md)) && 'cell_type' %in% colnames(md)) {
  md <- md %>% left_join(ct_broad_filt, by = 'cell_type')
  colData(cds)$cell_type_broad <- md$cell_type_broad
}

dim(md)
head(md[, seq_len(min(12, ncol(md)))])

In [ ]:
names(md)

In [ ]:
if ('cell_type' %in% names(md)) table(md$cell_type)[1:min(20, length(table(md$cell_type)))]
if ('perturbation' %in% names(md)) table(md$perturbation, useNA = 'ifany')
if ('expt' %in% names(md)) table(md$expt, useNA = 'ifany')
if ('timepoint' %in% names(md)) summary(md$timepoint)

## Quick Monocle plot

In [ ]:
plot_cells(cds, color_cells_by = 'cell_type_broad')

## Interactive UMAP plot with Plotly

In [ ]:
rd_names <- names(reducedDims(cds))
rd_names

In [ ]:
umap_key <- if ('UMAP' %in% rd_names) 'UMAP' else rd_names[1]
umap <- as.data.frame(reducedDims(cds)[[umap_key]])
colnames(umap)[1:2] <- c('dim1', 'dim2')
plot_df <- bind_cols(umap, md)

color_var <- if ('cell_type_broad' %in% names(plot_df)) 'cell_type_broad' else names(plot_df)[3]
hover_vars <- intersect(c('cell_type', 'cell_type_broad', 'perturbation', 'expt', 'timepoint', 'embryo_ID'), names(plot_df))

plot_ly(
  plot_df,
  x = ~dim1,
  y = ~dim2,
  color = plot_df[[color_var]],
  type = 'scatter',
  mode = 'markers',
  text = ~apply(plot_df[, hover_vars, drop = FALSE], 1, paste, collapse = '<br>'),
  hoverinfo = 'text'
)

## Filter cells interactively

In [ ]:
md_filt <- md %>%
  filter(!is.na(cell_type_broad)) %>%
  filter(is.na(perturbation) | perturbation %in% c('EtOH', 'DMSO', 'ctrl-inj', 'reference', 'ctrl-uninj', 'novehicle'))

dim(md_filt)
head(md_filt)

## Build embryo-level count set

In [ ]:
ccs <- new_cell_count_set(
  cds,
  sample_group = 'embryo_ID',
  cell_group = 'cell_type_broad'
)

colData(ccs)$pert_collapsed <- ifelse(
  colData(ccs)$perturbation %in% c('ctrl-uninj', 'reference', 'novehicle'),
  'ctrl',
  colData(ccs)$perturbation
)

ccs <- ccs[, !is.na(ccs$timepoint)]
ccs

In [ ]:
counts_df <- as.data.frame(as.matrix(counts(ccs)))
meta_df <- as.data.frame(colData(ccs))

dim(counts_df)
head(counts_df[, seq_len(min(8, ncol(counts_df)))])
head(meta_df[, seq_len(min(12, ncol(meta_df)))])

## Simple regression on embryo metadata

In [ ]:
lm_df <- meta_df %>%
  filter(!is.na(timepoint), !is.na(expt))

fit_time <- lm(timepoint ~ expt, data = lm_df)
summary(fit_time)

## Simple regression on one cell type count

In [ ]:
target_cell_type <- colnames(counts_df)[1]

reg_df <- meta_df %>%
  mutate(target_count = counts_df[[target_cell_type]]) %>%
  filter(!is.na(timepoint), !is.na(expt), !is.na(target_count))

fit_count <- lm(target_count ~ timepoint + expt, data = reg_df)
summary(fit_count)

In [ ]:
ggplot(reg_df, aes(x = timepoint, y = target_count, color = expt)) +
  geom_point(alpha = 0.7) +
  geom_smooth(method = 'lm', se = FALSE) +
  labs(title = paste('Embryo-level counts for', target_cell_type))

In [ ]:
#' returns a cell group x embryo count matrix 
#' @param ccs
#' @return a cell group by embryo count matrix
get_cell_count_wide <- function(ccs) {
  # cell group x embryo
  cell_count_wide = counts(ccs) %>%
    as.matrix %>%
    as.data.frame
  
  return(cell_count_wide)
}

#' takes in a ccs and outputs a matrix of
#' cell_type x embryo of proportions
#' @param cell_count_wide a cell group x embryo count matrix
#' @param pseudocount
get_prop_mat <- function(cell_count_wide,
                         pseudocount = 1) {
  
  prop_mat = cell_count_wide %>%
    rownames_to_column("cell_group") %>%
    tidyr::pivot_longer(-cell_group, names_to = "embryo", values_to = "count") %>%
    mutate(count = count + pseudocount) %>%
    group_by(embryo) %>%
    mutate(prop = (count)/sum(count)) %>%
    select(-count) %>%
    tidyr::pivot_wider(names_from = cell_group, values_from = prop) %>%
    tibble::column_to_rownames("embryo") %>%
    as.matrix
  
  return(prop_mat)
  
}

#' fits a drichlet model based off proportions
#' @param prop_mat
#' @param model_formula_str
#' @param trace
fit_drichlet <- function(prop_mat,
                         model_formula_str = "~ 1",
                         trace = FALSE) {
  
  
  model_formula_str = paste("prop_mat", model_formula_str)
  model_formula = as.formula(model_formula_str)
  
  dfit <- VGAM::vglm(model_formula,
                     data = as.data.frame(prop_mat),
                     family = "dirichlet",
                     trace = trace)
  
  return(dfit)
}


#' simulates a fake ccs based off wt data
#' @param ccs 
#' @param embryo_size how many cells per embryo
#' @param num_embryos how many embryo replicates
#' @param genotype
#' @param random.seed 
simulate_ccs = function(ccs, 
                        embryo_size = 1000,
                        random.seed = 111, 
                        genotype = "WT", 
                        num_embryos = NULL) {
  
  cell_count_wide = get_cell_count_wide(ccs)
  prop_mat = get_prop_mat(cell_count_wide)
  dfit = fit_drichlet(prop_mat)
  
  if (is.null(num_embryos)) {
    num_embryos = ncol(cell_count_wide)
  }
  num_cell_groups = nrow(cell_count_wide)
  
  num_sims = ceiling(num_embryos / ncol(cell_count_wide))
  
  dsim <- VGAM::simulate.vlm(dfit, nsim = num_sims, seed = random.seed)
  # this returns a (num_embryos x num_cell_groups) x num_sims matrix
  # make into a num_embryos x num_cell_groups x num_sims matrix 
  aaa <- array(unlist(dsim), c(num_embryos, num_cell_groups, num_sims))
  
  aaa_list = lapply(1:num_sims, function(i){
    # slice by dims
    aaa[,,i]
  })
  
  bbb = do.call(rbind, aaa_list) 
  bbb = bbb[1:num_embryos,]
  
  # how many total cells does each embryo have
  cell_totals = colSums(cell_count_wide)
  # get standard deviation from embryo totals in our experiment
  my.sd = sd(cell_totals)
  
  # simulate total cells
  # sim.total.cells = round(rnorm(num_embryos, mean = embryo_size, sd = my.sd))
  sim.total.cells = rpois(n = num_embryos, lambda = embryo_size)
  
  sim_props = reshape2::melt(bbb)
  colnames(sim_props) = c("embryo", "cell_group_number", "cell_prop")
  cell_group_names = rownames(cell_count_wide)
  df = data.frame("cell_group_number" = 1:length(cell_group_names),
                  "cell_group" = cell_group_names)
  sim_props = sim_props %>% left_join(df, by = "cell_group_number")
  
  tot_df = data.frame(unique(sim_props$embryo), sim.total.cells, stringsAsFactors = F)
  colnames(tot_df) = c('embryo', 'total_cells')
  
  
  # convert to counts
  sim_count_df = sim_props %>%
    # left_join(tot_df, by='embryo_unq') %>%
    left_join(tot_df, by='embryo') %>%
    mutate(count = round(total_cells*cell_prop)) %>%
    select(-c(total_cells, cell_prop, cell_group_number))
  
  sim_count_wide = sim_count_df %>%
    pivot_wider(names_from = "embryo", values_from = count) %>%
    tibble::column_to_rownames("cell_group") %>%
    as.matrix
  
  colnames(sim_count_wide) = paste0("embryo-", genotype, "-",colnames(sim_count_wide))
  # make a cell_count_cds
  covariates_df = data.frame("embryo" = colnames(sim_count_wide), 
                             genotype = genotype, 
                             effect = 0, 
                             embryo_size = embryo_size,
                             random.seed = random.seed, 
                             mutated = F)
  rownames(covariates_df) = covariates_df$embryo
  cell_count_cds = monocle3::new_cell_data_set(sim_count_wide,
                                               cell_metadata=covariates_df)
  sim_ccs = suppressMessages(
    methods::new("cell_count_set",
                 cell_count_cds,
                 cds = new_cell_data_set(hooke:::empty_sparse_matrix(format="C")), 
                 cds_coldata = tibble(), 
                 cds_reduced_dims = SimpleList(),
                 info = SimpleList())
  )
  
  return(sim_ccs)
  
}

#' mutates a ccs object
#'@param sim_ccs a simulated ccs object
#'@param cell_types a list of cell types to mutate, if null will randomly sample
#'@param num_cell_types integer how many cell types to randomly sample
#'@param effect_size how much to mutate cell types by
mutate_ccs = function(sim_ccs, 
                      cell_types = NULL,
                      num_cell_types = 10,
                      effect_size = 1.0, 
                      random.seed = 111) {
  
  sim_count_wide = get_cell_count_wide(sim_ccs)
  
  if (is.null(cell_types)) {
    set.seed(random.seed)
    all_cell_types = rownames(sim_count_wide)
    cell_types = sample(all_cell_types, num_cell_types)
  }
  
  sim_count_df = sim_count_wide %>% 
                 as.data.frame %>% 
                 rownames_to_column("cell_group") %>%
                 pivot_longer(-cell_group,
                             names_to = "embryo",
                             values_to = "count")
  
  sim_count_df = sim_count_df %>%
    mutate(count = ifelse(cell_group %in% cell_types,
                          round(count * (1-effect_size)),
                          count))
  
  sim_count_wide = sim_count_df %>%
    pivot_wider(names_from = "embryo",
                values_from = count) %>%
    tibble::column_to_rownames("cell_group") %>%
    as.matrix()
  
  covariates_df = colData(sim_ccs)
  
  covariates_df$effect_size = effect_size
  covariates_df$mutated_cell_types = list(cell_types)
  rownames(covariates_df) = colnames(sim_ccs) 
  colnames(sim_count_wide) = colnames(sim_ccs) 
  
  cell_count_cds = monocle3::new_cell_data_set(sim_count_wide,
                                               cell_metadata=covariates_df)
  
  # now turn it back into a cell count ccs
  mt_ccs = suppressMessages(
            methods::new("cell_count_set",
                        cell_count_cds,
                        cds = new_cell_data_set(hooke:::empty_sparse_matrix(format="C")), 
                        cds_coldata = tibble(), 
                        cds_reduced_dims = SimpleList(),
                        info = SimpleList())
  )
  return(mt_ccs)
  
}

#' combine two ccs objects
#' @param ccs_list 
combine_ccs = function(ccs_list) {
  
  comb_counts_list = lapply(ccs_list, function(ccs) {
    counts(ccs)
  })
  comb_counts = do.call(cbind, comb_counts_list)
  
  comb_coldata_list = lapply(ccs_list, function(ccs) {
    colData(ccs) %>% as.data.frame
  })
  
  comb_coldata = bind_rows(comb_coldata_list)
  
  cell_count_cds = monocle3::new_cell_data_set(comb_counts,
                                               cell_metadata=comb_coldata)
  
  comb_ccs = suppressMessages(
    methods::new("cell_count_set",
                 cell_count_cds,
                 cds = new_cell_data_set(hooke:::empty_sparse_matrix(format="C")), 
                 cds_coldata = tibble(), 
                 cds_reduced_dims = SimpleList(),
                 info = SimpleList())
  )
  return(comb_ccs)
}

#' @param ccs cell_count_set made from real data
#' @param embryo_size
#' @param random.seed
#' @param cell_types list of cell types to apply mutation to 
#' @param effect_size defaults to 1 
run_comparison = function(ccs, 
                          embryo_size = 1000, 
                          effect_size = 1,
                          num_embryos = 8,
                          cell_types = c(), 
                          random.seed = 111, 
                          num_threads = 4) {
  
  wt_ccs = simulate_ccs(ccs,
                        embryo_size = embryo_size,
                        num_embryos = num_embryos,
                        random.seed = random.seed, genotype = "WT")

  mt_ccs = simulate_ccs(ccs,
                        embryo_size = embryo_size,
                        num_embryos = num_embryos,
                        random.seed = random.seed*2, genotype = "MT")

  mt_ccs = mutate_ccs(mt_ccs, cell_types, effect_size = effect_size)

  ccs = combine_ccs(ccs_list = list(wt_ccs, mt_ccs))
  ccm = new_cell_count_model(ccs, 
                             main_model_formula_str = "~ genotype", 
                             penalize_by_distance = F, 
                             num_threads = num_threads, 
                             vhat_method = "bootstrap")
  
  cond_wt = estimate_abundances(ccm, tibble(genotype = "WT"))
  cond_mt = estimate_abundances(ccm, tibble(genotype = "MT"))
  cond_wt_v_mt_tbl = compare_abundances(ccm, cond_wt, cond_mt)
  return(cond_wt_v_mt_tbl)
}

## Notes

- Change `dataset_name` near the top to switch among `REF1`, `GENE1`, `GENE2`, `GENE3`.
- Set `use_hotfish2 <- TRUE` to load the Hotfish2 dataset instead.
- If you want the full multi-dataset batch analysis, use the original script. This notebook is intentionally smaller and interactive.
- If you want Hooke model fitting in the notebook too, that can be added as a second notebook or extra section.